# Aula 22 — Agent Orchestration Runtimes

**Quando um loop agente simples deixa de ser suficiente?**

Nas Aulas 20 e 21, construímos duas capacidades importantes:

```text
Aula 20
→ Agent Harness
→ loop agente controlado
→ decisão / autorização / execução

Aula 21
→ planning explícito
→ dependências
→ validação
→ replanning
```

Agora damos um passo diferente:

> **tornar estado, transições, persistência, interrupções e retomada parte explícita do runtime.**

Regra da aula:

```text
framework
≠
orchestration runtime

graph
≠
melhor arquitetura

orchestration runtime
=
coordenação explícita
+ estado observável
+ transições
+ checkpoints
+ interrupt / resume
+ recovery
```

A pergunta central será:

> **Quando essa camada adicional conquista sua complexidade?**

## 1. Objetivos

Ao final, você deverá conseguir:

- distinguir **Agent Harness** de **Orchestration Runtime**;
- modelar uma execução como grafo de estado;
- identificar nodes, edges e conditional routing;
- tornar shared state explícito;
- implementar checkpointing;
- explicar durable execution;
- implementar interrupt / resume;
- distinguir retry, resume e recovery;
- comparar código procedural com um graph runtime;
- medir overhead e utility da orquestração;
- reconhecer quando um framework não é necessário;
- ler LangGraph como implementação de referência, não como definição do conceito.

## 📘 Glossário da aula

- **[Runtime de orquestração](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#runtime-de-orquestração)**
- **[Checkpoint](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#checkpoint)**
- **[Execução durável](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#execução-durável)**
- **[Interrupção e retomada](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#interrupção-e-retomada)**

## 2. Harness, planner e orchestration runtime

Estas camadas podem coexistir, mas não são sinônimos.

```text
Decision Provider
→ propõe próxima ação

Agent Harness
→ controla o comportamento agente em runtime

Planner
→ representa plano, dependências e revisão

Orchestration Runtime
→ coordena estado, nodes, transitions,
  persistência, interrupt e resume
```

Um orchestration runtime pode fazer parte de um Agent Harness.

Da mesma forma:

```text
MCP
→ como capabilities são descobertas/acessadas

Orchestration Runtime
→ quando e em qual sequência essas capabilities executam
```

### 2.1 Arquitetura mínima, sem framework

Nesta aula usamos primeiro uma arquitetura vendor-neutral:

```text
Input / Goal
↓
Shared State
↓
Node
↓
Transition Policy
↓
Next Node
↓
Checkpoint
↓
Continue / Interrupt / Resume / Stop
```

Somente depois de compreender esse mecanismo faz sentido mapear conceitos para um framework como LangGraph.

In [ ]:
from dataclasses import dataclass, field, asdict
from copy import deepcopy
import pandas as pd

SCENARIOS = {
    "linear": {
        "request_valid": True,
        "lesson_ready": True,
        "approval_required": False,
        "approval_granted": True,
        "transient_failures": 0,
    },
    "branching": {
        "request_valid": True,
        "lesson_ready": False,
        "approval_required": False,
        "approval_granted": True,
        "transient_failures": 0,
    },
    "human_interrupt": {
        "request_valid": True,
        "lesson_ready": True,
        "approval_required": True,
        "approval_granted": False,
        "transient_failures": 0,
    },
    "transient_failure": {
        "request_valid": True,
        "lesson_ready": True,
        "approval_required": False,
        "approval_granted": True,
        "transient_failures": 1,
    },
    "durable_resume": {
        "request_valid": True,
        "lesson_ready": True,
        "approval_required": True,
        "approval_granted": False,
        "transient_failures": 0,
    },
}

print("Cenários:", list(SCENARIOS))

## 3. Shared state

O runtime precisa de um estado explícito.

Nesta versão, o estado é pequeno de propósito. O objetivo não é criar um framework genérico, mas tornar o mecanismo observável.

In [ ]:
@dataclass
class RuntimeState:
    scenario: str
    request_valid: bool = False
    lesson_ready: bool = False
    approval_required: bool = False
    approval_granted: bool = False
    release_note: bool = False
    published: bool = False
    node: str = "inspect"
    status: str = "running"
    reason: str = ""
    transitions: int = 0
    tool_calls: int = 0
    checkpoints: int = 0
    retries: int = 0
    resumes: int = 0
    human_interventions: int = 0
    transient_failures_remaining: int = 0
    trace: list = field(default_factory=list)

def make_state(name):
    cfg = SCENARIOS[name]
    return RuntimeState(
        scenario=name,
        request_valid=cfg["request_valid"],
        lesson_ready=cfg["lesson_ready"],
        approval_required=cfg["approval_required"],
        approval_granted=cfg["approval_granted"],
        transient_failures_remaining=cfg["transient_failures"],
    )

display(pd.DataFrame([asdict(make_state("linear"))]).drop(columns=["trace"]))

## 4. Nodes e transições

Cada node executa uma responsabilidade pequena.

A política de transição decide o próximo node a partir do estado observado.

```text
node
→ transforma estado

transition policy
→ escolhe próximo node
```

Isso é diferente de esconder toda a lógica em um único `while`.

In [ ]:
def record(state, event):
    state.trace.append({
        "node": state.node,
        "event": event,
        "status": state.status,
        "reason": state.reason,
    })

def inspect_node(state):
    record(state, "request inspected")
    state.tool_calls += 1
    return state

def prepare_node(state):
    state.release_note = True
    state.tool_calls += 1
    record(state, "release note prepared")
    return state

def approval_node(state):
    state.human_interventions += 1
    if not state.approval_granted:
        state.status = "interrupted"
        state.reason = "human_approval_required"
        record(state, "execution interrupted")
        return state
    record(state, "approval observed")
    return state

def publish_node(state):
    state.tool_calls += 1
    if state.transient_failures_remaining > 0:
        state.transient_failures_remaining -= 1
        state.status = "recoverable_error"
        state.reason = "transient_publish_failure"
        record(state, "transient failure")
        return state
    state.published = True
    state.status = "completed"
    state.reason = "published"
    record(state, "published")
    return state

def stop_node(state, reason):
    state.status = "completed"
    state.reason = reason
    record(state, reason)
    return state

NODES = {
    "inspect": inspect_node,
    "prepare": prepare_node,
    "approval": approval_node,
    "publish": publish_node,
}

In [ ]:
def next_node(state):
    if state.node == "inspect":
        if not state.request_valid:
            return "stop"
        if not state.lesson_ready:
            return "stop"
        return "prepare"

    if state.node == "prepare":
        return "approval" if state.approval_required else "publish"

    if state.node == "approval":
        return "publish" if state.approval_granted else "interrupt"

    if state.node == "publish":
        if state.status == "recoverable_error":
            return "retry"
        return "stop"

    return "stop"

for name in ["linear", "branching", "human_interrupt"]:
    s = make_state(name)
    inspect_node(s)
    print(name, "->", next_node(s))

## 5. Checkpoint

Um checkpoint é um **snapshot persistido do estado**.

Aqui usamos um store em memória para manter o laboratório determinístico e sem side effects externos.

Em produção, a persistência poderia estar em arquivo, banco, serviço de estado ou storage distribuído — mas o conceito é o mesmo.

In [ ]:
CHECKPOINT_STORE = {}

def save_checkpoint(run_id, state):
    # O contador faz parte do próprio estado persistido.
    state.checkpoints += 1
    CHECKPOINT_STORE[run_id] = deepcopy(state)
    return deepcopy(state)

def load_checkpoint(run_id):
    return deepcopy(CHECKPOINT_STORE[run_id])

demo = make_state("linear")
save_checkpoint("demo", demo)
restored = load_checkpoint("demo")

print("Checkpoint salvo:", "demo" in CHECKPOINT_STORE)
print("Estado restaurado:", restored.scenario, restored.node, restored.status)
print("checkpoints persistidos:", restored.checkpoints)

## 6. Graph runtime mínimo

Agora conectamos:

```text
shared state
+ nodes
+ transition policy
+ checkpoint
+ interrupt / resume
+ retry / recovery
```

O runtime abaixo continua sendo Python simples. Isso é intencional:

> **primeiro o mecanismo; depois o framework.**

In [ ]:
def run_graph(state, run_id, checkpoint_each_transition=True, max_transitions=20):
    while state.status == "running" and state.transitions < max_transitions:
        if state.node not in NODES:
            stop_node(state, "invalid_node")
            break

        NODES[state.node](state)

        if checkpoint_each_transition:
            save_checkpoint(run_id, state)

        if state.status == "interrupted":
            break

        nxt = next_node(state)

        if nxt == "stop":
            if not state.published:
                reason = "not_ready" if not state.lesson_ready else state.reason or "stopped"
                stop_node(state, reason)
            break

        if nxt == "retry":
            state.retries += 1
            state.status = "running"
            state.reason = ""
            nxt = "publish"

        state.node = nxt
        state.transitions += 1

    if state.transitions >= max_transitions and state.status == "running":
        stop_node(state, "transition_budget_exceeded")

    return state

for name in ["linear", "branching", "transient_failure"]:
    s = run_graph(make_state(name), f"graph-{name}")
    print(name, "=>", s.status, s.reason, "published=", s.published, "retries=", s.retries)

## 7. Interrupt e resume

No cenário de aprovação humana, a execução deve **parar de forma controlada**, preservar o estado e continuar depois da decisão.

```text
run
→ interrupt
→ checkpoint
→ tempo passa
→ decisão humana
→ load checkpoint
→ resume
```

Resume não significa começar tudo novamente.

In [ ]:
run_id = "human-demo"
state = run_graph(make_state("human_interrupt"), run_id)

print("Antes da aprovação:")
print(state.status, state.reason, state.node, "checkpoints=", state.checkpoints)

# Simula uma decisão externa posterior.
resumed = load_checkpoint(run_id)
resumed.approval_granted = True
resumed.status = "running"
resumed.reason = ""
resumed.resumes += 1

# O checkpoint foi salvo após o node de approval.
# Ao retomar, seguimos para publish.
resumed.node = "publish"
resumed = run_graph(resumed, run_id)

print("\nDepois da aprovação:")
print(resumed.status, resumed.reason, resumed.node, "published=", resumed.published, "resumes=", resumed.resumes)

### 7.1 Durable execution

A execução é **durável** quando o progresso relevante sobrevive ao processo que a iniciou.

Nosso laboratório simula isso reconstruindo a execução apenas a partir do checkpoint.

O valor aparece quando reiniciar do zero seria caro, arriscado ou impossível.

In [ ]:
run_id = "durable-demo"
first_process = run_graph(make_state("durable_resume"), run_id)

# Simula perda completa do objeto em memória do "processo 1".
del first_process

# "Processo 2" conhece somente o run_id e o checkpoint.
second_process = load_checkpoint(run_id)
second_process.approval_granted = True
second_process.status = "running"
second_process.reason = ""
second_process.resumes += 1
second_process.node = "publish"
second_process = run_graph(second_process, run_id)

print("Retomada a partir do checkpoint:")
print("status:", second_process.status)
print("published:", second_process.published)
print("resumes:", second_process.resumes)

## 8. Retry, resume e recovery não são a mesma coisa

```text
retry
→ repetir uma operação que falhou

resume
→ continuar uma execução pausada a partir de estado preservado

recovery
→ restaurar uma condição executável depois de uma falha
  e então decidir se deve retry, resume, compensar ou parar
```

No cenário `transient_failure`, o runtime identifica uma falha recuperável e repete somente o node de publicação.

In [ ]:
s = run_graph(make_state("transient_failure"), "recovery-demo")
display(pd.DataFrame(s.trace))
print("retries:", s.retries)
print("published:", s.published)

## 9. Baseline procedural

Para comparar de forma justa, implementamos a **mesma tarefa** sem graph runtime.

O baseline também usa estado explícito, mas coordena tudo por código procedural.

Isso evita um falso contraste:

```text
graph runtime
vs
código procedural propositalmente ruim
```

A comparação deve ser entre duas implementações razoáveis.

In [ ]:
def run_procedural(name):
    cfg = SCENARIOS[name]
    state = {
        "scenario": name,
        "published": False,
        "status": "running",
        "reason": "",
        "steps": 0,
        "tool_calls": 0,
        "retries": 0,
        "resumes": 0,
        "human_interventions": 0,
        "restart_cost": 0,
        "process_restart": False,
        "trace": [],
    }

    def step(event):
        state["steps"] += 1
        state["trace"].append(event)

    step("inspect")
    state["tool_calls"] += 1

    if not cfg["request_valid"]:
        state["status"] = "completed"
        state["reason"] = "invalid_request"
        return state

    if not cfg["lesson_ready"]:
        state["status"] = "completed"
        state["reason"] = "not_ready"
        return state

    step("prepare")
    state["tool_calls"] += 1

    if cfg["approval_required"] and not cfg["approval_granted"]:
        step("interrupt")
        state["human_interventions"] += 1

        if name == "durable_resume":
            # Aqui simulamos perda do processo local.
            # Sem checkpoint explícito, parte do progresso precisa ser reconstruída.
            state["process_restart"] = True
            state["restart_cost"] += 2
            step("reconstruct_after_process_restart")
        else:
            # Pausa curta no mesmo processo: há coordenação manual,
            # mas não perda do estado em memória.
            step("manual_resume_same_process")

        state["resumes"] += 1

    failures = cfg["transient_failures"]
    while True:
        step("publish")
        state["tool_calls"] += 1
        if failures > 0:
            failures -= 1
            state["retries"] += 1
            continue
        break

    state["published"] = True
    state["status"] = "completed"
    state["reason"] = "published"
    return state

for name in SCENARIOS:
    r = run_procedural(name)
    print(
        name,
        "=>", r["reason"],
        "steps=", r["steps"],
        "restart_cost=", r["restart_cost"],
        "process_restart=", r["process_restart"],
    )

## 10. Comparative Evidence Lab

Agora comparamos as duas arquiteturas nos cinco cenários.

As métricas são proxies didáticos, não benchmarks universais.

Medimos:

- task success;
- número de transições/steps;
- retries;
- resumes;
- intervenção humana;
- checkpoints;
- restart cost;
- cost proxy.

O **cost proxy** existe apenas para tornar overhead visível neste laboratório.

In [ ]:
def graph_metrics(name):
    run_id = f"compare-{name}"
    s = run_graph(make_state(name), run_id)

    if s.status == "interrupted":
        resumed = load_checkpoint(run_id)

        # durable_resume simula efetivamente outro processo:
        # a retomada usa somente o run_id + estado persistido.
        if name == "durable_resume":
            del s

        resumed.approval_granted = True
        resumed.status = "running"
        resumed.reason = ""
        resumed.resumes += 1
        resumed.node = "publish"
        s = run_graph(resumed, run_id)

    cost_proxy = (
        s.tool_calls
        + 0.35 * s.transitions
        + 0.20 * s.checkpoints
        + 0.50 * s.retries
        + 0.75 * s.human_interventions
    )

    return {
        "scenario": name,
        "architecture": "graph_runtime",
        "success": bool(s.published or s.reason == "not_ready"),
        "published": s.published,
        "transitions_or_steps": s.transitions,
        "retries": s.retries,
        "resumes": s.resumes,
        "human_interventions": s.human_interventions,
        "checkpoints": s.checkpoints,
        "process_restart": name == "durable_resume",
        "restart_cost": 0,
        "cost_proxy": round(cost_proxy, 2),
    }

def procedural_metrics(name):
    s = run_procedural(name)
    cost_proxy = (
        s["tool_calls"]
        + 0.20 * s["steps"]
        + 0.50 * s["retries"]
        + 0.75 * s["human_interventions"]
        + s["restart_cost"]
    )
    return {
        "scenario": name,
        "architecture": "procedural",
        "success": bool(s["published"] or s["reason"] == "not_ready"),
        "published": s["published"],
        "transitions_or_steps": s["steps"],
        "retries": s["retries"],
        "resumes": s["resumes"],
        "human_interventions": s["human_interventions"],
        "checkpoints": 0,
        "process_restart": s["process_restart"],
        "restart_cost": s["restart_cost"],
        "cost_proxy": round(cost_proxy, 2),
    }

rows = []
for name in SCENARIOS:
    rows.append(procedural_metrics(name))
    rows.append(graph_metrics(name))

evidence = pd.DataFrame(rows)
display(evidence)

In [ ]:
summary = evidence.pivot(
    index="scenario",
    columns="architecture",
    values=["success", "cost_proxy", "checkpoints", "restart_cost"],
)
display(summary)

### 10.1 Como interpretar a evidência

O objetivo **não** é provar que graph runtime é melhor.

A evidência deve permitir uma conclusão mais precisa:

- em fluxo simples, o runtime adiciona overhead sem melhorar o resultado;
- branching curto pode continuar tão simples quanto código procedural;
- uma interrupção humana no **mesmo processo** não é, por si só, evidência suficiente para durable execution;
- quando o processo é perdido e precisa continuar depois, checkpoint/resume reduz o custo de reconstrução;
- falha transitória permite localizar retry no node afetado, mas isso ainda pode custar mais do que um loop procedural pequeno;
- o ganho real aparece quando **coordenação, persistência e recuperação** passam a importar.

Portanto:

> **o runtime conquista sua complexidade quando capacidades de coordenação criam utility observável — e não simplesmente porque existe branching, retry ou um agente.**

## 11. Failure Lab — checkpoint incorreto

Persistência também cria uma nova superfície de falha.

Vamos simular um checkpoint corrompido que aponta para um node inexistente.

O objetivo é localizar a falha no runtime, não na decisão de negócio.

In [ ]:
bad = make_state("linear")
bad.node = "node_that_does_not_exist"
save_checkpoint("bad-checkpoint", bad)

restored = load_checkpoint("bad-checkpoint")
result = run_graph(restored, "bad-checkpoint")

print("status:", result.status)
print("reason:", result.reason)
print("published:", result.published)

## 12. Architecture Decision Lab

Escolha entre:

```text
procedural orchestration
graph runtime
```

Para cada caso, justifique por:

- quantidade de estados;
- branching;
- necessidade de persistência;
- necessidade de retomada;
- human-in-the-loop;
- duração do processo;
- reversibilidade;
- observabilidade;
- custo operacional;
- risco.

### Casos

**A.** Fluxo curto, linear, executado em segundos, sem aprovação humana.

**B.** Processo com vários branches, mas sempre concluído em uma única execução curta.

**C.** Processo que pode esperar horas por aprovação humana e deve continuar do ponto exato.

**D.** Processo longo, com falhas transitórias, checkpoints e necessidade de auditoria.

Pergunta central:

> **Em quais casos o graph runtime realmente conquistou sua complexidade?**

In [ ]:
from IPython.display import Markdown, display

class TILExercise:
    def __init__(self, hint_text, solution_text):
        self._hint_text = hint_text
        self._solution_text = solution_text

    def hint(self):
        display(Markdown(f"### Dica\n\n{self._hint_text}"))

    def solution(self):
        display(Markdown(f"### Solução de referência\n\n{self._solution_text}"))

q22_arch = TILExercise(
    "Branching, sozinho, não obriga um graph runtime. Observe principalmente persistência, retomada, duração e human-in-the-loop.",
    "A tende a procedural. B pode continuar procedural se o branching permanecer simples. C e D são os casos em que checkpoint, resume, recovery e auditabilidade começam a justificar um runtime explícito.",
)

print("Use q22_arch.hint() ou q22_arch.solution() somente quando desejar.")

## 13. E onde entra LangGraph?

Somente agora.

LangGraph é uma implementação possível dos mecanismos que acabamos de construir.

Mapeamento conceitual:

```text
conceito TIL          → possível primitiva de framework

shared state          → state schema
node                  → node/function
transition policy     → edge / conditional edge
checkpoint            → checkpointer
interrupt             → interrupt
resume                → retomada a partir do estado persistido
```

A aula **não depende de LangGraph** para funcionar.

Isso preserva o princípio:

> **Primeiro o mecanismo; depois o framework.**

Se amanhã o framework mudar, os conceitos continuam válidos.

## 14. Checagem de compreensão

Responda antes de abrir qualquer solução.

1. Qual a diferença entre Agent Harness e Orchestration Runtime?
2. O que torna um estado “shared state”?
3. Qual a diferença entre node e transition policy?
4. Por que checkpoint não é sinônimo de log?
5. Qual a diferença entre retry, resume e recovery?
6. O que durable execution acrescenta?
7. Por que branching sozinho não justifica um graph runtime?
8. Em que cenário o baseline procedural pode ser preferível?
9. Que nova superfície de falha nasce quando adicionamos persistência?
10. Por que LangGraph aparece somente no final da aula?
11. Que evidência justificaria migrar de código procedural para um runtime explícito?
12. Como esta aula estende o princípio “complexidade precisa ser conquistada por evidência”?

## 15. Reprodutibilidade

```text
Internet OFF
GPU OFF
sem API externa
sem side effects reais
runtime local e determinístico
LangGraph opcional, não requerido
```

O laboratório principal depende apenas de Python e pandas disponíveis no ambiente do curso.

## 16. Síntese

```text
goal
→ shared state
→ node
→ transition
→ checkpoint
→ continue / interrupt / resume
→ recovery
→ stop
```

A Aula 20 mostrou **como controlar um comportamento agente**.

A Aula 21 mostrou **como representar e validar um plano explícito**.

A Aula 22 mostra **como coordenar a execução ao longo do tempo e do estado**.

> **Um orchestration runtime não torna o sistema automaticamente melhor ou mais agente. Ele se justifica quando estado, branching, persistência, retomada, recovery ou supervisão se tornam complexos o suficiente para exigir uma camada explícita de coordenação.**


---

## Continue no TIL

← **[Anterior: Aula 21 — Planning and Goal Decomposition](https://www.kaggle.com/code/pedrogentil/til-21-planning-and-goal-decomposition)** &nbsp;&nbsp;|&nbsp;&nbsp; 🏠 **[Apresentação do curso](https://www.kaggle.com/code/pedrogentil/text-intelligence-lab-applied-ai-engineering)**

TIL · Pedro Gentil Regato de Oliveira Soares · Conteúdo: CC BY 4.0 · Código: MIT
